In [3]:
%load_ext autoreload
%autoreload 2

from datetime import date, datetime, timedelta
from functools import partial
from typing import Dict, Optional

import numpy as np
import polars as pl
from tqdm import tqdm
import time

from okx.store import OrderbookStore
from okx.recipes.forwards import build_forwards_pchip, build_forwards_kalman, assign_forwards
from okx.recipes.options import prepare_options
from evaluation.forwards_eval import evaluate_parity, summarize_parity, evaluate_pillar_fit, evaluate_loeo

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite"
)

In [132]:
store.clear_cache()

Cleared all caches


In [5]:
# Shared parameters
inst_family = 'BTC-USD'

# Scenario (a): Full month with binning
def make_dates(n):
    return [date(2025, 9, 1) + timedelta(days=i) for i in range(n)]
dates_month = make_dates(30)
binning_month = '5m'

# Scenario (b): Single day with option timestamps
dates_day = [date(2025, 9, 2)]
binning_day = None  # Will use unique_times from OPTIONS

def benchmark_fn(df: pl.DataFrame) -> None:
    print(f"Benchmarking {df.shape[0]} rows")
    for col in df.columns:
        print(f"{col:<10} sorted: {df.select(col).equals(df.select(col).sort(col))}")


In [6]:
lf_futures_1 = store.get(
    inst_family=inst_family,
    inst_type='FUTURES',
    dates=make_dates(10),
    depth=0,
    binning='5m',
    features=['trim', 'strip', 'bin_ff', 'sink_bins'],
    verbose=True,
    benchmark=True,
    batch_days=5
)

[store] Getting BTC-USD/FUTURES for 10 dates (depth=0, 5m binning, 4 features)


Processing batches:   0%|          | 0/2 [00:00<?, ?it/s]

  - applied 'trim'
  [benchmark] trim: 0.292s
  - applied 'strip'
  [benchmark] strip: 0.000s
  - applied 'bin_ff'
  [benchmark] bin_ff: 0.381s
  - applied 'sink_bins'
  [benchmark] sink_bins: 0.000s
  - applied 'trim'
  [benchmark] trim: 0.189s
  - applied 'strip'
  [benchmark] strip: 0.000s
  - applied 'bin_ff'
  [benchmark] bin_ff: 0.259s
  - applied 'sink_bins'
  [benchmark] sink_bins: 0.000s
  [benchmark] Direct build: 1.133s


In [7]:
lf_futures_2 = store.get(
    inst_family=inst_family,
    inst_type='FUTURES',
    dates=make_dates(10),
    depth=0,
    binning='5m',
    features=['trim', 'strip', 'bin_ff', 'sink_bins'],
    verbose=True,
    benchmark=True,
    batch_days=5
)

[store] Getting BTC-USD/FUTURES for 10 dates (depth=0, 5m binning, 4 features)


Processing batches:   0%|          | 0/2 [00:00<?, ?it/s]

  - applied 'trim'
  [benchmark] trim: 0.217s
  - applied 'strip'
  [benchmark] strip: 0.000s
  - applied 'bin_ff'
  [benchmark] bin_ff: 0.338s
  - applied 'sink_bins'
  [benchmark] sink_bins: 0.000s
  - applied 'trim'
  [benchmark] trim: 0.153s
  - applied 'strip'
  [benchmark] strip: 0.000s
  - applied 'bin_ff'
  [benchmark] bin_ff: 0.259s
  - applied 'sink_bins'
  [benchmark] sink_bins: 0.000s
  [benchmark] Direct build: 0.973s


In [8]:
df1 = lf_futures_1.collect()
df2 = lf_futures_2.collect()

# Goal: Print all rows from df whose timeMs appears a number of times != 7 (i.e. where not all symbols are present at that timeMs).
# Solution:

metrics = df1.group_by('symbol').agg(
    pl.col('timeMs').n_unique().alias('n_bins'),
    pl.col('timeMs').min().cast(pl.Datetime("ms")).alias('min_time'),
    pl.col('timeMs').max().cast(pl.Datetime("ms")).alias('max_time')
)
print(metrics.sort('n_bins', descending=True))

df1 = df1.sort(['symbol', 'timeMs'])
df2 = df2.sort(['symbol', 'timeMs'])

print(df1.height)
print(df2.height)

diff_df1 = df1.join(df2, on=df1.columns, how='anti')
print(diff_df1)

diff_df2 = df2.join(df1, on=df2.columns, how='anti').with_columns(pl.col('timeMs').cast(pl.Datetime("ms")).alias('timeMs'))
print(diff_df2)









shape: (8, 4)
┌───────────────────┬────────┬─────────────────────┬─────────────────────┐
│ symbol            ┆ n_bins ┆ min_time            ┆ max_time            │
│ ---               ┆ ---    ┆ ---                 ┆ ---                 │
│ str               ┆ u32    ┆ datetime[ms]        ┆ datetime[ms]        │
╞═══════════════════╪════════╪═════════════════════╪═════════════════════╡
│ BTC-USD-260626.OK ┆ 2880   ┆ 2025-09-01 00:05:00 ┆ 2025-09-11 00:00:00 │
│ BTC-USD-250912.OK ┆ 2880   ┆ 2025-09-01 00:05:00 ┆ 2025-09-11 00:00:00 │
│ BTC-USD-251226.OK ┆ 2880   ┆ 2025-09-01 00:05:00 ┆ 2025-09-11 00:00:00 │
│ BTC-USD-250926.OK ┆ 2880   ┆ 2025-09-01 00:05:00 ┆ 2025-09-11 00:00:00 │
│ BTC-USD-251031.OK ┆ 2880   ┆ 2025-09-01 00:05:00 ┆ 2025-09-11 00:00:00 │
│ BTC-USD-260327.OK ┆ 2880   ┆ 2025-09-01 00:05:00 ┆ 2025-09-11 00:00:00 │
│ BTC-USD-250919.OK ┆ 1440   ┆ 2025-09-06 00:05:00 ┆ 2025-09-11 00:00:00 │
│ BTC-USD-250905.OK ┆ 1249   ┆ 2025-09-01 00:05:00 ┆ 2025-09-05 08:05:00 │
└──────────

In [9]:
unique_times = store.get(
    inst_family=inst_family,
    inst_type='OPTION',
    dates=make_dates(3),
    depth=0,
    features=['trim', 'strip'],
    verbose=True,
    batch_days=3
).select('timeMs').unique().sort('timeMs').collect().to_series().to_list()

[store] Getting BTC-USD/OPTION for 3 dates (depth=0, provided timestamps, 2 features)
  - applied 'trim'
  - applied 'strip'


In [10]:
lf_futures_3 = store.get(
    inst_family=inst_family,
    inst_type='FUTURES',
    dates=make_dates(3),
    depth=0,
    unique_times=unique_times,
    features=['trim', 'strip', 'dedupe', 'bin_ff', 'sink_bins'],
    verbose=True,
    benchmark=True,
    batch_days=1
)

[store] Getting BTC-USD/FUTURES for 3 dates (depth=0, provided timestamps, 5 features)


Processing batches:   0%|          | 0/3 [00:00<?, ?it/s]

  - applied 'trim'
  [benchmark] trim: 0.049s
  - applied 'strip'
  [benchmark] strip: 0.000s
  - applied 'dedupe'
  [benchmark] dedupe: 0.696s
  - applied 'bin_ff'
  [benchmark] bin_ff: 3.265s
  - applied 'sink_bins'
  [benchmark] sink_bins: 0.054s
  - applied 'trim'
  [benchmark] trim: 0.063s
  - applied 'strip'
  [benchmark] strip: 0.000s
  - applied 'dedupe'
  [benchmark] dedupe: 0.753s
  - applied 'bin_ff'
  [benchmark] bin_ff: 3.186s
  - applied 'sink_bins'
  [benchmark] sink_bins: 0.052s
  - applied 'trim'
  [benchmark] trim: 0.052s
  - applied 'strip'
  [benchmark] strip: 0.000s
  - applied 'dedupe'
  [benchmark] dedupe: 0.622s
  - applied 'bin_ff'
  [benchmark] bin_ff: 2.526s
  - applied 'sink_bins'
  [benchmark] sink_bins: 0.043s
  [benchmark] Direct build: 12.910s


In [133]:
options_fitted = prepare_options(
    store=store,
    inst_family=inst_family,
    dates=make_dates(3),
    forwards_recipe=build_forwards_kalman,
    paired=True,
    verbose=True,
    batch_days=1
)

Preparing paired options data for BTC-USD for 3 dates using build_forwards_kalman
[store] Getting BTC-USD/OPTION for 3 dates (depth=1, provided timestamps, 6 features)


Processing batches:   0%|          | 0/3 [00:00<?, ?it/s]

  - applied 'trim'
  - applied 'strip'
  - applied 'nullify'
  - applied 'dedupe'
  - applied 'tenor'
  - applied 'parse_option'
  - applied 'trim'
  - applied 'strip'
  - applied 'nullify'
  - applied 'dedupe'
  - applied 'tenor'
  - applied 'parse_option'
  - applied 'trim'
  - applied 'strip'
  - applied 'nullify'
  - applied 'dedupe'
  - applied 'tenor'
  - applied 'parse_option'
 - Time taken to fetch options: 0:00:21.677059
[store] Getting BTC-USD/SPOT for 3 dates (depth=0, provided timestamps, 4 features)


Processing batches:   0%|          | 0/3 [00:00<?, ?it/s]

  - applied 'trim'
  - applied 'strip'
  - applied 'dedupe'
  - applied '<lambda>'
  - applied 'trim'
  - applied 'strip'
  - applied 'dedupe'
  - applied '<lambda>'
  - applied 'trim'
  - applied 'strip'
  - applied 'dedupe'
  - applied '<lambda>'
 - Time taken to fetch spot: 0:00:00.774969
 - Time taken for numeraire conversion: 0:00:00.000111
Assigning kalman forwards to 3 dates using kalman forwards
 - Time taken to collect data: 0:00:03.301268
[store] Getting derived data via recipe 'build_forwards_kalman' for 3 dates
Building Kalman-filtered Nelson-Siegel forwards for BTC-USD with provided timestamps
Constructing pillars for BTC-USD with swap and futures using provided timestamps
  [benchmark] trim: 0.083s
  [benchmark] strip: 0.000s
  [benchmark] dedupe: 0.921s
  [benchmark] rel_spread: 0.058s
  [benchmark] tenor: 0.046s
  [benchmark] log: 0.038s
  [benchmark] Cache building: 1.932s
  [benchmark] Cache loading: 0.001s
  [benchmark] trim: 0.259s
  [benchmark] strip: 0.000s
  [ben

Kalman filter:   0%|          | 0/1000274 [00:00<?, ?it/s]

 - Time taken to run Kalman filter: 0:00:45.845905
 - Time taken to convert states to Polars: 0:00:00.162174
 - Total time taken to build Kalman forwards: 0:01:09.259517
 - Time taken to fetch forwards: 0:01:12.725253
 - Time taken to build forward lookup: 0:00:04.051396


Matching forwards: 100%|██████████| 1000274/1000274 [00:15<00:00, 63410.63it/s]

 - Time taken to match forwards: 0:00:15.798951, matched 1,840,327 / 1,840,327 rows
 - Total time taken to assign forwards: 0:01:32.575600
 - Time taken for forwards assignment and moneyness calculation: 0:01:32.622129
 - Total time taken for options data preparation: 0:01:55.074268


In [ ]:
metrics = lf_futures_3.group_by('timeMs').agg(
    pl.len().alias('n_pillars')
).group_by('n_pillars').agg(
    pl.len().alias('n_timestamps')
).sort('n_pillars').collect()

print(metrics)




